# 🎲 Monte Carlo Simulation for Options Pricing
**Tools:** `numpy` · `scipy` · `matplotlib` · `pandas`

This notebook simulates 10,000+ random stock price paths using **Geometric Brownian Motion (GBM)**,
prices European Call and Put options from the simulation, then cross-checks the result against the
closed-form **Black-Scholes formula**.

---
### Key parameters you can change
| Symbol | Meaning | Default |
|--------|---------|--------|
| `S0` | Current stock price | $100 |
| `K` | Strike price | $105 |
| `T` | Time to expiration (years) | 1.0 |
| `r` | Risk-free interest rate | 5% |
| `sigma` | Annual volatility | 20% |
| `N_SIMS` | Number of simulated paths | 100,000 |
| `N_STEPS` | Daily time steps per path | 252 |

## ⚙️ Step 1 — Install & import libraries

In [ ]:
# scipy is pre-installed on Colab; numpy and matplotlib are too
# No extra installs needed!
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)   # reproducible results

print('✅  All libraries loaded')

## 🎛️ Step 2 — Define parameters
All parameters live here. Change them and re-run all cells to see how the price changes.

In [ ]:
# ── Option & market parameters ─────────────────────────────────
S0      = 100.0    # Current stock price ($)
K       = 105.0    # Strike price ($)
T       = 1.0      # Time to expiration (years)
r       = 0.05     # Annual risk-free rate (5%)
sigma   = 0.20     # Annual volatility (20%)

# ── Simulation parameters ──────────────────────────────────────
N_SIMS  = 100_000  # Number of price paths
N_STEPS = 252      # Trading days in a year (time steps per path)

dt = T / N_STEPS   # Length of each time step (fraction of a year)

print(f'Parameters set:')
print(f'  Stock price S₀ = ${S0:.0f}  |  Strike K = ${K:.0f}')
print(f'  Maturity T = {T} yr  |  r = {r*100:.1f}%  |  σ = {sigma*100:.0f}%')
print(f'  Simulations: {N_SIMS:,}  |  Time steps: {N_STEPS}')
print(f'  dt = {dt:.6f} years ({dt*365:.2f} calendar days)')

## 📐 Step 3 — Geometric Brownian Motion (GBM)

Each stock price path follows GBM — the standard model for random stock prices:

$$S_{t+\Delta t} = S_t \cdot \exp\left[(r - \tfrac{1}{2}\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\, Z\right]$$

where **Z ~ N(0,1)** is a random draw (the randomness). The term $(r - \frac{1}{2}\sigma^2)\Delta t$
is the **drift** — the expected upward pull — and $\sigma\sqrt{\Delta t}\, Z$ is the **diffusion** — the random noise.

In [ ]:
# ── Simulate all paths at once using NumPy vectorisation ───────
# Z shape: (N_STEPS, N_SIMS) — one random number per step per path
Z = np.random.standard_normal((N_STEPS, N_SIMS))

# Daily log-returns
log_returns = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

# Cumulative product along the time axis → price paths
# shape: (N_STEPS + 1, N_SIMS)
price_paths = np.zeros((N_STEPS + 1, N_SIMS))
price_paths[0] = S0
price_paths[1:] = S0 * np.exp(np.cumsum(log_returns, axis=0))

print(f'✅  Simulated {N_SIMS:,} price paths, each with {N_STEPS} steps')
print(f'    price_paths matrix shape: {price_paths.shape}')
print(f'    Min final price : ${price_paths[-1].min():.2f}')
print(f'    Max final price : ${price_paths[-1].max():.2f}')
print(f'    Mean final price: ${price_paths[-1].mean():.2f}  (expected: ${S0 * np.exp(r * T):.2f})')

## 📊 Step 4 — Visualise sample price paths
We plot 200 sample paths out of 100,000 so the chart remains readable.

In [ ]:
N_PLOT = 200
time_axis = np.linspace(0, T, N_STEPS + 1)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(time_axis, price_paths[:, :N_PLOT], color='#2196F3', alpha=0.07, linewidth=0.6)
ax.axhline(K, color='#FF5722', linewidth=1.5, linestyle='--', label=f'Strike price K = ${K}')
ax.axhline(S0, color='#4CAF50', linewidth=1.2, linestyle=':', label=f'Starting price S₀ = ${S0}')
ax.plot(time_axis, price_paths[:, :N_PLOT].mean(axis=1), color='#0D47A1',
        linewidth=2, label='Mean path')
ax.set_title(f'Sample GBM Price Paths ({N_PLOT} of {N_SIMS:,} shown)  |  σ={sigma*100:.0f}%  r={r*100:.0f}%')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Stock Price ($)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('gbm_paths.png', dpi=150, bbox_inches='tight')
plt.show()

## 💵 Step 5 — Calculate option payoffs at expiration

At expiration, the payoff of each option depends on the **final stock price S_T** vs the strike K:

| Option | Payoff formula | Intuition |
|--------|---------------|----------|
| **Call** | max(S_T − K, 0) | You profit if the stock ends above the strike |
| **Put**  | max(K − S_T, 0) | You profit if the stock ends below the strike |

In [ ]:
# Final prices across all simulated paths
S_T = price_paths[-1]   # shape: (N_SIMS,)

# Payoffs at expiration
call_payoffs = np.maximum(S_T - K, 0)
put_payoffs  = np.maximum(K - S_T, 0)

print(f'Call payoffs  —  mean: ${call_payoffs.mean():.4f}  |  % in-the-money: {(call_payoffs > 0).mean()*100:.1f}%')
print(f'Put  payoffs  —  mean: ${put_payoffs.mean():.4f}  |  % in-the-money: {(put_payoffs > 0).mean()*100:.1f}%')

## 🔮 Step 6 — Monte Carlo option prices (discount back to today)

The option price today = **average payoff** discounted at the risk-free rate:

$$\text{Price} = e^{-rT} \cdot \frac{1}{N}\sum_{i=1}^{N} \text{Payoff}_i$$

We also compute a **95% confidence interval** to show how accurate the simulation is.

In [ ]:
discount = np.exp(-r * T)

# Monte Carlo prices
mc_call = discount * call_payoffs.mean()
mc_put  = discount * put_payoffs.mean()

# 95% confidence intervals (standard error of the mean)
call_se = call_payoffs.std() / np.sqrt(N_SIMS)
put_se  = put_payoffs.std()  / np.sqrt(N_SIMS)
z95     = 1.96

print('Monte Carlo option prices')
print('=' * 45)
print(f'  Call price : ${mc_call:.4f}  '
      f'95% CI [{discount*(call_payoffs.mean()-z95*call_se):.4f}, '
      f'{discount*(call_payoffs.mean()+z95*call_se):.4f}]')
print(f'  Put  price : ${mc_put:.4f}  '
      f'95% CI [{discount*(put_payoffs.mean()-z95*put_se):.4f}, '
      f'{discount*(put_payoffs.mean()+z95*put_se):.4f}]')
print('=' * 45)

## ✅ Step 7 — Black-Scholes closed-form solution (ground truth)

Black-Scholes gives an exact analytical price for European options:

$$C = S_0 \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$
$$P = K e^{-rT} \cdot N(-d_2) - S_0 \cdot N(-d_1)$$

where:
$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

and $N(\cdot)$ is the **cumulative normal distribution** function.

In [ ]:
def black_scholes(S, K, T, r, sigma):
    """Returns (call_price, put_price) using the Black-Scholes formula."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    put  = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return call, put, d1, d2

bs_call, bs_put, d1, d2 = black_scholes(S0, K, T, r, sigma)

print(f'Black-Scholes solution')
print('=' * 45)
print(f'  d1 = {d1:.4f}   N(d1) = {norm.cdf(d1):.4f}')
print(f'  d2 = {d2:.4f}   N(d2) = {norm.cdf(d2):.4f}')
print(f'  Call price : ${bs_call:.4f}')
print(f'  Put  price : ${bs_put:.4f}')
print('=' * 45)

## 📋 Step 8 — Side-by-side comparison

In [ ]:
comparison = pd.DataFrame({
    'Method'      : ['Monte Carlo', 'Black-Scholes', 'Difference ($)', 'Difference (%)'],
    'Call price'  : [
        f'${mc_call:.4f}',
        f'${bs_call:.4f}',
        f'${abs(mc_call - bs_call):.4f}',
        f'{abs(mc_call - bs_call)/bs_call*100:.3f}%'
    ],
    'Put price'   : [
        f'${mc_put:.4f}',
        f'${bs_put:.4f}',
        f'${abs(mc_put - bs_put):.4f}',
        f'{abs(mc_put - bs_put)/bs_put*100:.3f}%'
    ]
})
display(comparison.style.hide(axis='index'))

# Quick sanity check: put-call parity  C - P = S0 - K*e^(-rT)
parity_lhs = mc_call - mc_put
parity_rhs = S0 - K * np.exp(-r * T)
print(f'\nPut-call parity check:')
print(f'  C - P = ${parity_lhs:.4f}   |   S₀ - Ke^(-rT) = ${parity_rhs:.4f}   |   '
      f'Diff = ${abs(parity_lhs - parity_rhs):.4f} ✅')

## 📉 Step 9 — Visualise payoff distribution & convergence

Two plots:
1. **Payoff distribution** — histogram of all final payoffs across 100k paths
2. **Convergence** — how the Monte Carlo price stabilises as we add more simulations

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── Plot 1: Call payoff distribution ───────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
in_money = call_payoffs[call_payoffs > 0]
ax1.hist(in_money, bins=80, color='#2196F3', alpha=0.8, edgecolor='none')
ax1.axvline(mc_call / discount, color='#FF5722', linewidth=2,
            label=f'Mean payoff = ${mc_call/discount:.2f}')
ax1.set_title(f'Call payoff distribution\n({(call_payoffs>0).mean()*100:.1f}% of paths expire in-the-money)')
ax1.set_xlabel('Payoff at expiration ($)')
ax1.set_ylabel('Frequency')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.25)

# ── Plot 2: Put payoff distribution ────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
in_money_p = put_payoffs[put_payoffs > 0]
ax2.hist(in_money_p, bins=80, color='#9C27B0', alpha=0.8, edgecolor='none')
ax2.axvline(mc_put / discount, color='#FF5722', linewidth=2,
            label=f'Mean payoff = ${mc_put/discount:.2f}')
ax2.set_title(f'Put payoff distribution\n({(put_payoffs>0).mean()*100:.1f}% of paths expire in-the-money)')
ax2.set_xlabel('Payoff at expiration ($)')
ax2.set_ylabel('Frequency')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)

# ── Plot 3: Convergence of call price ──────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
checkpoints = np.logspace(2, np.log10(N_SIMS), 60).astype(int)
running_call = [np.exp(-r*T) * call_payoffs[:n].mean() for n in checkpoints]
ax3.semilogx(checkpoints, running_call, color='#2196F3', linewidth=1.6)
ax3.axhline(bs_call, color='#FF5722', linewidth=1.5, linestyle='--',
            label=f'Black-Scholes = ${bs_call:.4f}')
ax3.set_title('Monte Carlo call price convergence')
ax3.set_xlabel('Number of simulations')
ax3.set_ylabel('Estimated call price ($)')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.25)

# ── Plot 4: Convergence of put price ───────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
running_put = [np.exp(-r*T) * put_payoffs[:n].mean() for n in checkpoints]
ax4.semilogx(checkpoints, running_put, color='#9C27B0', linewidth=1.6)
ax4.axhline(bs_put, color='#FF5722', linewidth=1.5, linestyle='--',
            label=f'Black-Scholes = ${bs_put:.4f}')
ax4.set_title('Monte Carlo put price convergence')
ax4.set_xlabel('Number of simulations')
ax4.set_ylabel('Estimated put price ($)')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.25)

fig.suptitle('Monte Carlo Options Pricing — Results', fontsize=14, fontweight='bold')
plt.savefig('monte_carlo_options_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊  Chart saved as monte_carlo_options_results.png')

## 🔬 Step 10 — Sensitivity analysis (How does price change with σ and S₀?)

In [ ]:
# Black-Scholes price across a range of spot prices and volatilities
spot_range  = np.linspace(60, 140, 200)
sigma_range = [0.10, 0.20, 0.30, 0.40]
colors      = ['#1565C0', '#2196F3', '#FF9800', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for sig, col in zip(sigma_range, colors):
    calls = [black_scholes(s, K, T, r, sig)[0] for s in spot_range]
    puts  = [black_scholes(s, K, T, r, sig)[1] for s in spot_range]
    axes[0].plot(spot_range, calls, color=col, label=f'σ = {sig*100:.0f}%')
    axes[1].plot(spot_range, puts,  color=col, label=f'σ = {sig*100:.0f}%')

for ax, title in zip(axes, ['Call price vs stock price', 'Put price vs stock price']):
    ax.axvline(K, color='black', linewidth=0.8, linestyle=':', alpha=0.6)
    ax.set_xlabel('Stock price S₀ ($)')
    ax.set_ylabel('Option price ($)')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    ax.text(K + 1, ax.get_ylim()[1] * 0.95, f'K=${K}', fontsize=8, color='black', alpha=0.6)

fig.suptitle('Black-Scholes sensitivity to volatility', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 (Optional) Download all charts

In [ ]:
from google.colab import files
files.download('gbm_paths.png')
files.download('monte_carlo_options_results.png')
files.download('sensitivity_analysis.png')